In [46]:
import pandas as pd

In [47]:
caminho = "/home/gabriel/pessoal/faculdade/mineracao/t2/AgeOfMythology_work.csv"
df = pd.read_csv(caminho, dtype="string", keep_default_na=False, encoding="utf-8")

In [48]:
df

,Unit,Age,Cost,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,LOS,Train time,Bonus,Other,Class
0,Pegasus,1,50F 2Fv,2,--,--,--,0,140,50,50,99,5,18,6,,,Myth Unit
1,Centaur,2,200W 15Fv,3,--,12,--,12,220,25,35,80,5,20,19,"Myth units, -Heroes",,Myth Unit
2,Cyclops,2,250F 22Fv,4,15,--,12,2,600,40,50,80,3.2,16,20,Myth units,,Myth Unit
3,Minotaur,2,200F 16Fv,4,15,--,10,2,300,60,50,80,4,20,20,Myth units,,Myth Unit
4,Hydra,3,250F 28Fv,5,20,--,10,3,800,60,40,80,4,16,20,"Myth units, -Heroes",,Myth Unit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,Longboat,2,100W 50G,2,--,6,--,12,290,30%,20%,10%,6,24,14,,Transports,Ship
79,Drakkar,3,100W 50G,2,20,--,--,1,240,20%,20%,75%,7,16,12,,,Ship
80,Dragon Boat,3,100W 100G,3,--,--,6,18,480,10%,50%,10%,4.8,24,17,-Buildings,,Ship
81,Kraken,3,300F 25Fv,5,15,--,5,1,900,50%,70%,80%,6,22,10,,,Myth Unit


In [ ]:

import pandas as pd

RECURSOS = {
    "F": "cost_food",
    "W": "cost_wood",
    "G": "cost_gold",
    "Fv": "cost_favor",
}

COLUNAS_NUMERICAS = [
    "age", "pop", "dmg_hack", "dmg_pierce", "dmg_crush",
    "range", "hp", "armor_hack", "armor_pierce",
    "armor_crush", "speed", "los", "train_time",
]


def extrair_custos(valor):
    if pd.isna(valor) or valor in ("", "--"):
        return {nome: pd.NA for nome in RECURSOS.values()}

    custos = {nome: 0 for nome in RECURSOS.values()}

    for item in valor.replace(",", " ").split():
        recurso = next(
            (r for r in RECURSOS if item.endswith(r)),
            None
        )

        if recurso is None:
            raise ValueError(f"Custo inválido: {item}")

        quantidade = item.removesuffix(recurso)

        if not quantidade.isdigit():
            raise ValueError(f"Custo inválido: {item}")

        custos[RECURSOS[recurso]] += int(quantidade)

    return custos


def limpar_dataset(df) -> pd.DataFrame:
    df = df.replace("", pd.NA)

    # Renomeia colunas
    df = df.rename(columns={
        "Unit": "unit",
        "Age": "age",
        "Cost": "cost_raw",
        "Pop": "pop",
        "DmgHack": "dmg_hack",
        "DmgPierce": "dmg_pierce",
        "DmgCrush": "dmg_crush",
        "Range": "range",
        "HP": "hp",
        "ArmorHack": "armor_hack",
        "ArmorPierce": "armor_pierce",
        "ArmorCrush": "armor_crush",
        "Speed": "speed",
        "LOS": "los",
        "Train time": "train_time",
        "Bonus": "bonus",
        "Other": "other",
        "Class": "unit_class",
    })

    # Corrige erros de escrita
    for coluna in ["unit_class", "bonus"]:
        df[coluna] = df[coluna].str.replace(
            "Infantary", "Infantry", regex=False
        )

    # Extrai custos
    custos = df["cost_raw"].apply(extrair_custos)
    custos = pd.DataFrame(custos.tolist(), index=df.index)

    for coluna in custos.columns:
        df[coluna] = custos[coluna].astype("Int64")

    # Remove marcações dos danos
    for coluna in ["dmg_hack", "dmg_pierce", "dmg_crush"]:
        df[coluna] = df[coluna].str.replace("*", "", regex=False)

    # Corrige HP
    df["hp"] = df["hp"].str.removesuffix("_")

    # Remove símbolo de porcentagem das armaduras
    for coluna in ["armor_hack", "armor_pierce", "armor_crush"]:
        df[coluna] = df[coluna].str.removesuffix("%")

    # Converte colunas numéricas
    for coluna in COLUNAS_NUMERICAS:
        substituicoes = {
            "--": -1 if coluna.startswith("dmg_") else pd.NA,
            "none": pd.NA,
        }

        df[coluna] = pd.to_numeric(
            df[coluna].replace(substituicoes),
            errors="raise"
        )

    # Define tipos
    colunas_inteiras = [
        "age", "pop", "range", "hp",
        "armor_hack", "armor_pierce", "armor_crush",
        "los", "train_time",
    ]

    colunas_decimais = [
        "dmg_hack", "dmg_pierce", "dmg_crush", "speed",
    ]

    df[colunas_inteiras] = df[colunas_inteiras].astype("Int64")
    df[colunas_decimais] = df[colunas_decimais].astype("Float64")

    # Remove duplicatas
    df = df.drop_duplicates().reset_index(drop=True)

    # Valida valores
    for coluna in ["armor_hack", "armor_pierce", "armor_crush"]:
        if (df[coluna].dropna() > 100).any():
            raise ValueError(f"Armadura inválida: {coluna}")

    # Organiza colunas
    ordem = [
        "unit", "unit_class", "age",
        "cost_food", "cost_wood", "cost_gold", "cost_favor",
        "pop", "hp",
        "dmg_hack", "dmg_pierce", "dmg_crush",
        "armor_hack", "armor_pierce", "armor_crush",
        "range", "speed", "los", "train_time",
        "bonus", "other",
    ]

    return df[ordem]


In [50]:
limpar_dataset(df).to_csv("teste.csv")